# Caso 04: VueltaFit — Gimnasios: ¿Qué Plan Deja Más Dinero?
## Día 3 (miércoles): La Primera Grieta — El Anual Renueva Peor

**Objetivo del día:**
Partir la tasa agregada de renovación (52%) para descubrir la realidad bajo la superficie:
1. **La Primera Grieta:** Demostrar que el plan anual (el «socio de oro» de Growth) renueva sustancialmente peor que el mensual.
2. **El Sesgo de Supervivencia:** Probar que el hueco de socios que «pagan pero no vienen» está compuesto mayoritariamente por socios anuales que prepagaron y desertaron.
3. **El Interrogatorio de Sospechosos:** Descartar rigurosamente explicaciones alternativas (Canal de captación, Sede y Edad) antes de plantear la causa conductual del problema.


In [1]:
import duckdb
import pandas as pd
import os

# Inicializar conexión DuckDB y registrar vistas de datos limpios
con = duckdb.connect()
DATA_CLEAN = os.path.join('data', 'clean')

for table in ['socios', 'suscripciones', 'checkins', 'clases', 'reservas', 'pagos', 'sedes']:
    csv_path = os.path.join(DATA_CLEAN, f'{table}_clean.csv').replace('\\', '/')
    con.execute(f"CREATE VIEW v_{table} AS SELECT * FROM read_csv_auto('{csv_path}')")

print('Conexión y vistas DuckDB listas.')


Conexión y vistas DuckDB listas.


### 1. La Primera Grieta: Tasa de Renovación por Tipo de Plan

Calculamos la renovación sobre contratos vencidos a 30 de junio de 2024 usando la regla de negocio:
* El socio arranca una nueva suscripción dentro de los **30 días posteriores al vencimiento**.


In [2]:
q_renov_plan = '''
WITH vencidas AS (
    SELECT *
    FROM v_suscripciones
    WHERE TRY_CAST(fecha_fin AS DATE) <= DATE '2024-06-30'
),
con_renov AS (
    SELECT 
        v.*,
        EXISTS (
            SELECT 1 
            FROM v_suscripciones s2 
            WHERE s2.socio_id = v.socio_id 
              AND s2.suscripcion_id != v.suscripcion_id
              AND TRY_CAST(s2.fecha_inicio AS DATE) >= TRY_CAST(v.fecha_fin AS DATE)
              AND TRY_CAST(s2.fecha_inicio AS DATE) <= TRY_CAST(v.fecha_fin AS DATE) + INTERVAL 30 DAY
        ) AS renovo
    FROM vencidas v
)
SELECT 
    tipo_plan,
    COUNT(*) AS total_vencidas,
    SUM(CASE WHEN renovo THEN 1 ELSE 0 END) AS renovadas,
    ROUND(AVG(CASE WHEN renovo THEN 1.0 ELSE 0.0 END) * 100, 2) AS tasa_renovacion_pct,
    ROUND((1.0 - AVG(CASE WHEN renovo THEN 1.0 ELSE 0.0 END)) * 100, 2) AS tasa_churn_pct
FROM con_renov
GROUP BY tipo_plan
ORDER BY tasa_renovacion_pct DESC;
'''
df_renov_plan = con.execute(q_renov_plan).df()
print('=== Tasa de Renovación y Churn por Tipo de Plan ===')
display(df_renov_plan)


=== Tasa de Renovación y Churn por Tipo de Plan ===


,tipo_plan,total_vencidas,renovadas,tasa_renovacion_pct,tasa_churn_pct
0,mensual,2227,1328.0,59.63,40.37
1,anual,1530,629.0,41.11,58.89


**Análisis de la Primera Grieta (`59,63%` vs `41,11%`):**
* Al abrir el $52,09\%$ global por tipo de plan, el ranking de Rubén se invierte por completo:
  * **Plan Mensual:** de `2.227` contratos vencidos, renuevan **`1.328` ($59,63\%$ de renovación; $40,37\%$ de *churn*)**.
  * **Plan Anual:** de `1.530` contratos vencidos, solo renuevan **`629` ($41,11\%$ de renovación; $58,89\%$ de *churn*)**.
* El supuesto «socio de oro» anual sufre una **caída de $-18,52\text{ puntos porcentuales}$ en renovación** frente al mensual (casi 6 de cada 10 socios anuales se marchan al terminar el año).

---
### 2. El Sesgo de Supervivencia: ¿Dónde se esconden los `1.185` socios que pagan pero no vienen?
Desglosamos por `tipo_plan` a los **`4.370` socios con suscripción activa** según hayan registrado o no algún check-in en las últimas 4 semanas antes del corte (`30/06/2024`).

In [3]:
q_hueco_plan = '''
WITH socios_activos AS (
    SELECT DISTINCT socio_id, tipo_plan
    FROM v_suscripciones
    WHERE activa
),
socios_que_vienen AS (
    SELECT DISTINCT socio_id
    FROM v_checkins
    WHERE TRY_CAST(fecha_hora AS DATE) >= DATE '2024-06-30' - INTERVAL 28 DAY
      AND TRY_CAST(fecha_hora AS DATE) <= DATE '2024-06-30'
)
SELECT 
    sa.tipo_plan,
    COUNT(sa.socio_id) AS total_activos_pagan,
    COUNT(sqv.socio_id) AS vienen_ultimas_4sem,
    COUNT(sa.socio_id) - COUNT(sqv.socio_id) AS pagan_pero_no_vienen,
    ROUND((COUNT(sa.socio_id) - COUNT(sqv.socio_id)) * 100.0 / COUNT(sa.socio_id), 2) AS pct_inactivos_del_plan
FROM socios_activos sa
LEFT JOIN socios_que_vienen sqv ON sa.socio_id = sqv.socio_id
GROUP BY sa.tipo_plan
ORDER BY pct_inactivos_del_plan DESC;
'''
df_hueco_plan = con.execute(q_hueco_plan).df()
print('=== Desglose del Hueco de Inactivos por Tipo de Plan ===')
display(df_hueco_plan)


=== Desglose del Hueco de Inactivos por Tipo de Plan ===


,tipo_plan,total_activos_pagan,vienen_ultimas_4sem,pagan_pero_no_vienen,pct_inactivos_del_plan
0,anual,2759,1779,980,35.52
1,mensual,1611,1406,205,12.73


**Análisis del Cementerio de Inactivos (Sesgo de Supervivencia del Prepago):**
* **En el Plan Anual:** de `2.759` socios con contrato activo, **`980` socios ($35,52\%$) no pisan el gimnasio desde hace más de un mes**. Más de un tercio de la cartera anual activa ya ha desertado en la práctica, pero sigue contando como «activa» porque pagó 12 meses por adelantado.
* **En el Plan Mensual:** de `1.611` socios activos, solo **`205` ($12,73\%$)** están inactivos (quien deja de ir en el plan mensual cancela rápido el recibo; quien pagó el año entero permanece atrapado contablemente hasta el mes 12).
* En conjunto, **el $82,70\%$ de todos los socios «fantasma» de VueltaFit (`980` de `1.185`) pertenecen al plan anual**.

---
### 3. El Interrogatorio de Sospechosos: Descartar Edad, Canal de Captación y Sede
Antes de señalar al contrato o al hábito, comprobamos si la caída del plan anual se debe a una variable demográfica o de captación:
* **Sospechoso 1 (`Edad`):** ¿Son los anuales más jóvenes o más mayores?
* **Sospechoso 2 (`Canal de captación`):** ¿Entran por canales de menor calidad?
* **Sospechoso 3 (`Ciudad / Sede`):** ¿Se concentran en ciudades con peor servicio?

In [4]:
# 3A. Sospechoso A: Edad por Plan
q_edad = '''
WITH primer_plan AS (
    SELECT socio_id, tipo_plan
    FROM (
        SELECT socio_id, tipo_plan, ROW_NUMBER() OVER (PARTITION BY socio_id ORDER BY fecha_inicio) AS rn
        FROM v_suscripciones
    ) WHERE rn = 1
)
SELECT 
    p.tipo_plan,
    COUNT(s.socio_id) AS n_socios,
    ROUND(AVG(s.edad), 2) AS edad_media,
    ROUND(MEDIAN(s.edad), 2) AS edad_mediana
FROM primer_plan p
JOIN v_socios s USING (socio_id)
WHERE NOT s.es_outlier_edad
GROUP BY p.tipo_plan;
'''
df_edad = con.execute(q_edad).df()
print('=== Sospechoso 1: Edad por Plan ===')
display(df_edad)

# 3B. Sospechoso B: Renovación por Canal de Captación
q_canal = '''
WITH vencidas AS (
    SELECT s.*, so.canal_captacion
    FROM v_suscripciones s
    JOIN v_socios so USING (socio_id)
    WHERE TRY_CAST(s.fecha_fin AS DATE) <= DATE '2024-06-30'
),
con_renov AS (
    SELECT 
        v.*,
        EXISTS (
            SELECT 1 
            FROM v_suscripciones s2 
            WHERE s2.socio_id = v.socio_id 
              AND s2.suscripcion_id != v.suscripcion_id
              AND TRY_CAST(s2.fecha_inicio AS DATE) >= TRY_CAST(v.fecha_fin AS DATE)
              AND TRY_CAST(s2.fecha_inicio AS DATE) <= TRY_CAST(v.fecha_fin AS DATE) + INTERVAL 30 DAY
        ) AS renovo
    FROM vencidas v
)
SELECT 
    canal_captacion,
    COUNT(*) AS total_vencidas,
    ROUND(AVG(CASE WHEN renovo THEN 1.0 ELSE 0.0 END) * 100, 2) AS tasa_renovacion_pct
FROM con_renov
GROUP BY canal_captacion
ORDER BY tasa_renovacion_pct DESC;
'''
df_canal = con.execute(q_canal).df()
print('=== Sospechoso 2: Renovación por Canal de Captación ===')
display(df_canal)

# 3C. Sospechoso C: Renovación por Sede / Ciudad
q_sede = '''
WITH vencidas AS (
    SELECT s.*, se.ciudad
    FROM v_suscripciones s
    JOIN v_socios so USING (socio_id)
    JOIN v_sedes se ON so.sede_id = se.sede_id
    WHERE TRY_CAST(s.fecha_fin AS DATE) <= DATE '2024-06-30'
),
con_renov AS (
    SELECT 
        v.*,
        EXISTS (
            SELECT 1 
            FROM v_suscripciones s2 
            WHERE s2.socio_id = v.socio_id 
              AND s2.suscripcion_id != v.suscripcion_id
              AND TRY_CAST(s2.fecha_inicio AS DATE) >= TRY_CAST(v.fecha_fin AS DATE)
              AND TRY_CAST(s2.fecha_inicio AS DATE) <= TRY_CAST(v.fecha_fin AS DATE) + INTERVAL 30 DAY
        ) AS renovo
    FROM vencidas v
)
SELECT 
    ciudad,
    COUNT(*) AS total_vencidas,
    ROUND(AVG(CASE WHEN renovo THEN 1.0 ELSE 0.0 END) * 100, 2) AS tasa_renovacion_pct
FROM con_renov
GROUP BY ciudad
ORDER BY tasa_renovacion_pct DESC;
'''
df_sede = con.execute(q_sede).df()
print('=== Sospechoso 3: Renovación por Sede / Ciudad ===')
display(df_sede)


=== Sospechoso 1: Edad por Plan ===


,tipo_plan,n_socios,edad_media,edad_mediana
0,mensual,5336,34.58,34.0
1,anual,3655,34.77,35.0


=== Sospechoso 2: Renovación por Canal de Captación ===


,canal_captacion,total_vencidas,tasa_renovacion_pct
0,promo,533,54.97
1,referral,281,54.09
2,web,1072,52.24
3,walkin,157,52.23
4,campana,273,52.01
5,recomendacion,626,51.76
6,walk-in,301,50.17
7,online,514,49.22


=== Sospechoso 3: Renovación por Sede / Ciudad ===


,ciudad,total_vencidas,tasa_renovacion_pct
0,Barcelona,908,54.30
1,Sevilla,414,53.62
2,Málaga,392,53.32
3,Zaragoza,526,51.90
4,Madrid,1020,50.29
5,Valencia,497,49.70


**Análisis de los 3 Sospechosos (Todos tienen coartada):**
1. **Edad idéntica:** La edad media (excluyendo los 9 outliers) es de **$34,58$ años en mensual** (mediana $34,0$) y **$34,77$ años en anual** (mediana $35,0$). No existe sesgo demográfico.
2. **Canal de captación uniforme:** Las tasas de renovación se mueven en una banda estrecha entre el **$49,22\%$ (`online`) y el $54,97\%$ (`promo`)**, muy lejos de explicar los $18,52\text{ pp}$ de brecha entre planes.
3. **Ciudad / Sede uniforme:** Las 6 ciudades renuevan prácticamente igual en el agregado, desde el **$49,70\%$ en Valencia** hasta el **$54,30\%$ en Barcelona**.

---
### 4. Prueba Definitiva: Cruce Plan $\times$ Ciudad
Verificamos si la ventaja de renovación del plan mensual frente al anual se mantiene **dentro de cada una de las 6 ciudades**.

In [5]:
q_cruce = '''
WITH vencidas AS (
    SELECT s.*, se.ciudad
    FROM v_suscripciones s
    JOIN v_socios so USING (socio_id)
    JOIN v_sedes se ON so.sede_id = se.sede_id
    WHERE TRY_CAST(s.fecha_fin AS DATE) <= DATE '2024-06-30'
),
con_renov AS (
    SELECT 
        v.*,
        EXISTS (
            SELECT 1 
            FROM v_suscripciones s2 
            WHERE s2.socio_id = v.socio_id 
              AND s2.suscripcion_id != v.suscripcion_id
              AND TRY_CAST(s2.fecha_inicio AS DATE) >= TRY_CAST(v.fecha_fin AS DATE)
              AND TRY_CAST(s2.fecha_inicio AS DATE) <= TRY_CAST(v.fecha_fin AS DATE) + INTERVAL 30 DAY
        ) AS renovo
    FROM vencidas v
)
SELECT 
    ciudad,
    ROUND(AVG(CASE WHEN tipo_plan = 'mensual' AND renovo THEN 1.0 WHEN tipo_plan = 'mensual' THEN 0.0 END) * 100, 1) AS renov_mensual_pct,
    ROUND(AVG(CASE WHEN tipo_plan = 'anual' AND renovo THEN 1.0 WHEN tipo_plan = 'anual' THEN 0.0 END) * 100, 1) AS renov_anual_pct,
    ROUND(AVG(CASE WHEN tipo_plan = 'mensual' AND renovo THEN 1.0 WHEN tipo_plan = 'mensual' THEN 0.0 END) * 100 - 
          AVG(CASE WHEN tipo_plan = 'anual' AND renovo THEN 1.0 WHEN tipo_plan = 'anual' THEN 0.0 END) * 100, 1) AS brecha_puntos
FROM con_renov
GROUP BY ciudad
ORDER BY brecha_puntos DESC;
'''
df_cruce = con.execute(q_cruce).df()
print('=== Brecha Mensual vs Anual por Sede (Puntos Porcentuales) ===')
display(df_cruce)


=== Brecha Mensual vs Anual por Sede (Puntos Porcentuales) ===


,ciudad,renov_mensual_pct,renov_anual_pct,brecha_puntos
0,Barcelona,63.5,40.7,22.8
1,Málaga,62.8,40.8,22.0
2,Madrid,58.6,38.0,20.7
3,Sevilla,60.2,42.5,17.7
4,Zaragoza,57.3,44.7,12.6
5,Valencia,54.1,43.4,10.7


### Conclusiones Día 3 (Miércoles) — El Veredicto del Detective

1. **La Grieta es Estructural (`59,63%` vs `41,11%`):** El plan mensual supera al anual en **$+18,52\text{ pp}$ de renovación** ($59,6\%$ frente a $41,1\%$).
2. **El Sesgo de Supervivencia (`35,52%` vs `12,73%`):** El **$35,5\%$ de los socios anuales activos (`980` socios)** son «socios fantasma» que llevan más de un mes sin acudir al club, casi el triple que en el plan mensual ($12,7\%$, `205` socios). El prepago maquilla el abandono durante 11 meses hasta que explota de golpe al vencer el año.
3. **Descarte de Variables de Ruido (Edad, Canal y Sede):**
   * **Edad:** $34,8$ años (anual) vs $34,6$ años (mensual).
   * **Canal de captación:** todos los canales se sitúan entre $49,2\%$ y $55,0\%$.
   * **Ciudad / Sede:** todas las ciudades se sitúan entre $49,7\%$ y $54,3\%$, y **en las 6 ciudades el plan mensual supera al anual** (desde $+10,7\text{ pp}$ en Valencia hasta $+22,8\text{ pp}$ en Barcelona).
4. **Hipótesis Conductual para el Día 4:** Si no importa quién eres, ni por qué canal entraste, ni en qué ciudad entrenas, la diferencia debe estar en **el hábito que construyes durante tus primeras 3 semanas en el gimnasio**.